<a href="https://colab.research.google.com/github/TetianaMar-888/Python_for_ds_tasks/blob/main/Tetiana_Marinoshenko_HW_2_4_%D0%90%D0%BB%D0%B3%D0%BE%D1%80%D0%B8%D1%82%D0%BC%D0%B8_%D0%B1%D1%83%D1%81%D1%82%D0%B8%D0%BD%D0%B3%D1%83.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням алгоритмів бустингу: XGBoost та LightGBM, а також використаємо бібліотеку HyperOpt для оптимізації гіперпараметрів.

0. Зчитайте дані `train.csv` в змінну `raw_df` та скористайтесь наведеним кодом нижче аби розділити дані на трнувальні та валідаційні і розділити дані на ознаки з матириці Х та цільову змінну. Назви змінних `train_inputs, train_targets, train_inputs, train_targets` можна змінити на ті, які Вам зручно.

  Наведений скрипт - частина отриманого мною скрипта для обробки даних. Ми тут не викнуємо масштабування та обробку категоріальних змінних, бо хочемо це делегувати алгоритмам, які будемо використовувати. Якщо щось не розумієте в наведених скриптах, рекомендую розібратись: навичка читати код - важлива складова роботи в машинному навчанні.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

In [3]:
raw_df = pd.read_csv('/content/drive/MyDrive/Machine_Learning/bank-customer-churn-prediction-dlu-course-c/train.csv')

In [4]:
raw_df.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15779985.0,Nwankwo,678.0,France,Male,29.0,4.0,0.00,3.0,1.0,0.0,180626.36,0.0
1,1,15650086.0,Ch'in,687.0,France,Female,34.0,1.0,0.00,2.0,0.0,1.0,63736.17,0.0
2,2,15733602.0,Thompson,682.0,France,Female,52.0,6.0,0.00,3.0,0.0,0.0,179655.87,1.0
3,3,15645794.0,Macleod,753.0,Germany,Male,44.0,6.0,83347.25,2.0,1.0,0.0,161407.48,0.0
4,4,15633840.0,Hsia,544.0,Germany,Female,55.0,0.0,107747.57,1.0,1.0,0.0,176580.86,1.0


In [5]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import Tuple, Dict, Any


def split_train_val(df: pd.DataFrame, target_col: str, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into training and validation sets.

    Args:
        df (pd.DataFrame): The raw dataframe.
        target_col (str): The target column for stratification.
        test_size (float): The proportion of the dataset to include in the validation split.
        random_state (int): Random state for reproducibility.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Training and validation dataframes.
    """
    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df[target_col])
    return train_df, val_df


def separate_inputs_targets(df: pd.DataFrame, input_cols: list, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Separate inputs and targets from the dataframe.

    Args:
        df (pd.DataFrame): The dataframe.
        input_cols (list): List of input columns.
        target_col (str): Target column.

    Returns:
        Tuple[pd.DataFrame, pd.Series]: DataFrame of inputs and Series of targets.
    """
    inputs = df[input_cols].copy()
    targets = df[target_col].copy()
    return inputs, targets

In [6]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
! pip freeze | grep xgboost

xgboost==3.2.0


1. В тренувальному та валідаційному наборі перетворіть категоріальні ознаки на тип `category`. Можна це зробити двома способами:
 1. `df[col_name].astype('category')`, як було продемонстровано в лекції
 2. використовуючи метод `pd.Categorical(df[col_name])`

In [7]:
train_df, val_df = split_train_val(raw_df, target_col='Exited')

In [8]:
cat_features = train_df.select_dtypes(include=['object']).columns
train_df[cat_features] = train_df[cat_features].astype('category')
val_df[cat_features] = val_df[cat_features].astype('category')

In [9]:
train_df.shape, val_df.shape

((12000, 14), (3000, 14))

2. Навчіть на отриманих даних модель `XGBoostClassifier`. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів XGBoostClassifier - тут https://xgboost.readthedocs.io/en/stable/parameter.html#global-config

  **Важливо:** зробіть такі налаштування `XGBoostClassifier` аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Можна також, якщо працюєте в Google Colab, увімкнути можливість використання GPU (`Runtime -> Change runtime type -> T4 GPU`) і встановити параметр `device='cuda'` в `XGBoostClassifier` для пришвидшення тренування бустинг моделі.
  
  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням DecisionTrees раніше. Чи вийшло покращити якість?

In [10]:
# 1. Визначаємо назву таргету та вхідні колонки
target_col = 'Exited'
input_cols = [col for col in train_df.columns if col != target_col]

# 2. Розділяємо inputs / targets (використовуємо функцію з попереднього кроку)
X_train, y_train = separate_inputs_targets(train_df, input_cols, target_col)
X_val,   y_val   = separate_inputs_targets(val_df,   input_cols, target_col)

# 3. Модель
xgb_clf = XGBClassifier(
    max_depth=3,
    n_estimators=100,
    learning_rate=0.1,
    enable_categorical=True,   # обробка категоріальних ознак
    missing=np.nan,            # обробка пропущених значень
    device='cuda',
    random_state=42,
)

xgb_clf.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False,
)

# 4. Передбачення ймовірностей (потрібно для AUROC)
train_proba = xgb_clf.predict_proba(X_train)[:, 1]
val_proba   = xgb_clf.predict_proba(X_val)[:, 1]

# 5. AUROC
train_auroc = roc_auc_score(y_train, train_proba)
val_auroc   = roc_auc_score(y_val,   val_proba)

print(f"Train AUROC: {train_auroc:.4f}")
print(f"Val   AUROC: {val_auroc:.4f}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:06:07] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:06:07] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


Train AUROC: 0.9621
Val   AUROC: 0.9348


#1
Train AUROC: 0.9621

Val   AUROC: 0.9348

#2
Модель не перенавчена

#3
В порівнянні з DecisionTrees дає кращі результати на тренувальному та валідаційному наборах

3. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `XGBoostClassifier` з лекції знайдіть оптимальні значення гіперпараметрів `XGBoostClassifier` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **20**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. В ній ми маємо задати loss - це може будь-яка метрика, але бажано використовувтаи ту, яка цільова в вашій задачі. Чим менший лосс - тим ліпша модель на думку hyperopt. Тож, тут нам треба задати loss - негативне значення AUROC. В лекції ми натомість використовували Accuracy.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_clf` модель `XGBoostClassifier` з найкращими гіперпараметрами
    - навчіть модель `final_clf`
    - оцініть якість моделі `final_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (2) цього завдання?

In [11]:
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

In [12]:
def objective(params):
  clf = xgb.XGBClassifier(
      n_estimators=int(params ['n_estimators']),
      max_depth=int(params ['max_depth']),
      learning_rate=params ['learning_rate'],
      min_child_weight=params['min_child_weight'],
      subsample=params['subsample'],
      colsample_bytree=params['colsample_bytree'],
      gamma=params['gamma'],
      reg_alpha=params['reg_alpha'],
      reg_lambda=params['reg_lambda'],
      enable_categorical=True,
      missing=np.nan,
      device='cuda',
      random_state=42,
      early_stopping_rounds=10,
  )

  clf.fit(
      X_train, y_train,
      eval_set=[(X_val, y_val)],
      verbose=False)
  pred = clf.predict_proba(X_val)
  auroc = roc_auc_score(y_val, pred[:, 1])

  return {'loss': -auroc, 'status': STATUS_OK}

  #Гіперпараметри
space = {
    'n_estimators': hp.quniform('n_estimators', 50, 500, 25),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1.0),
    'gamma': hp.uniform('gamma', 0, 0.5),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 1)
}

# Оптимізація
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=20, trials=trials)

# Перетворення значень гіперпараметрів у кінцеві типи
best['n_estimators'] = int(best['n_estimators'])
best['max_depth'] = int(best['max_depth'])
best['min_child_weight'] = int(best['min_child_weight'])

print("Найкращі гіперпараметри: ", best)

# Навчання фінальної моделі з найкращими гіперпараметрами
final_clf = xgb.XGBClassifier(
    n_estimators=best['n_estimators'],
    learning_rate=best['learning_rate'],
    max_depth=best['max_depth'],
    min_child_weight=best['min_child_weight'],
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    gamma=best['gamma'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda'],
    enable_categorical=True,
    missing=np.nan,
    device='cuda',
)

final_clf.fit(X_train, y_train)
train_proba_final = final_clf.predict_proba(X_train)[:, 1]
val_proba_final   = final_clf.predict_proba(X_val)[:, 1]
train_auroc_final = roc_auc_score(y_train, train_proba_final)
val_auroc_final   = roc_auc_score(y_val,   val_proba_final)
print(f"Final model — Train AUROC: {train_auroc_final:.4f}")
print(f"Final model — Val   AUROC: {val_auroc_final:.4f}")

# Порівняння з базовою моделлю
print(f"\nБазова модель — Train AUROC: {train_auroc:.4f}")
print(f"Базова модель — Val   AUROC: {val_auroc:.4f}")

print(f"\nПокращення Val AUROC: {val_auroc_final - val_auroc:+.4f}")

  0%|          | 0/20 [00:00<?, ?trial/s, best loss=?]

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [17:06:13] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [17:06:13] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()



  5%|▌         | 1/20 [00:01<00:26,  1.39s/trial, best loss: -0.9304081212703202]

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [17:06:14] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [17:06:14] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()



 10%|█         | 2/20 [00:03<00:34,  1.91s/trial, best loss: -0.935060703751972] 

...

 90%|█████████ | 18/20 [00:17<00:02,  1.00s/trial, best loss: -0.9373187461417107]

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [17:06:30] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [17:06:30] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()



100%|██████████| 20/20 [00:18<00:00,  1.08trial/s, best loss: -0.9373187461417107]

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [17:06:31] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [17:06:31] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:06:31] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:06:31] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iterati


Найкращі гіперпараметри:  {'colsample_bytree': np.float64(0.7232943578913913), 'gamma': np.float64(0.18639676258901627), 'learning_rate': np.float64(0.0685224518024569), 'max_depth': 5, 'min_child_weight': 4, 'n_estimators': 425, 'reg_alpha': np.float64(0.9778299170721878), 'reg_lambda': np.float64(0.919714013474974), 'subsample': np.float64(0.9485109221392221)}
Final model — Train AUROC: 0.9818
Final model — Val   AUROC: 0.9336

Базова модель — Train AUROC: 0.9621
Базова модель — Val   AUROC: 0.9348

Покращення Val AUROC: -0.0012


Покращення моделі після тюнінгу не відбулося.

4. Навчіть на наших даних модель LightGBM. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів LightGBM - тут https://lightgbm.readthedocs.io/en/latest/Parameters.html

  **Важливо:** зробіть такі налаштування LightGBM аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Аби передати категоріальні колонки в LightGBM - необхідно виявити їх індекси і передати в параметрі `cat_feature=cat_feature_indexes`

  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням XGBoostClassifier раніше. Чи вийшло покращити якість?

In [13]:
%%bash
sudo apt-get update
sudo apt-get install -y build-essential cmake git wget unzip
sudo apt-get install -y libboost-dev libboost-system-dev libboost-filesystem-dev
sudo apt-get install -y libboost-iostreams-dev libboost-program-options-dev libboost-regex-dev
sudo apt-get install -y libboost-thread-dev libboost-chrono-dev libboost-date-time-dev
sudo apt-get install -y libboost-atomic-dev libboost-serialization-dev
sudo apt-get install -y python3-pip

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,870 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.9 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 6.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
de

In [14]:
%%bash
sudo apt-get install -y ocl-icd-libopencl1 clinfo
sudo apt-get install -y nvidia-opencl-dev opencl-headers

Reading package lists...
Building dependency tree...
Reading state information...
clinfo is already the newest version (3.0.21.02.21-1).
ocl-icd-libopencl1 is already the newest version (2.2.14-3).
ocl-icd-libopencl1 set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
Reading package lists...
Building dependency tree...
Reading state information...
nvidia-opencl-dev is already the newest version (11.5.1-1ubuntu1).
The following NEW packages will be installed:
  opencl-headers
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 1,754 B of archives.
After this operation, 12.3 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 opencl-headers all 3.0~2022.01.04-1 [1,754 B]
Fetched 1,754 B in 0s (5,417 B/s)
Selecting previously unselected package opencl-headers.
(Reading database ... 118761 files and directories currently installed.)
Preparing to unpack .../opencl-headers_3.0~

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


In [15]:
%%bash
git clone --recursive https://github.com/microsoft/LightGBM
cd LightGBM
mkdir build
cd build
cmake -DUSE_CUDAP=1 ..
make -j4

Submodule path 'external_libs/compute': checked out '36350b7de849300bd3d72a05d8bf890ca405a014'
Submodule path 'external_libs/eigen': checked out '3147391d946bb4b6c68edd901f2add6ac1f31f8c'
Submodule path 'external_libs/fast_double_parser': checked out '252029ddac664370bdda3f0761675785d92a1573'
Submodule path 'external_libs/fast_double_parser/benchmarks/dependencies/abseil-cpp': checked out 'd936052d32a5b7ca08b0199a6724724aea432309'
Submodule path 'external_libs/fast_double_parser/benchmarks/dependencies/double-conversion': checked out 'f4cb2384efa55dee0e6652f8674b05763441ab09'
Submodule path 'external_libs/fmt': checked out '8303d140a1a11f19b982a9f664bbe59a1ccda3f4'
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI i

Cloning into 'LightGBM'...
Submodule 'include/boost/compute' (https://github.com/boostorg/compute) registered for path 'external_libs/compute'
Submodule 'eigen' (https://gitlab.com/libeigen/eigen.git) registered for path 'external_libs/eigen'
Submodule 'external_libs/fast_double_parser' (https://github.com/lemire/fast_double_parser.git) registered for path 'external_libs/fast_double_parser'
Submodule 'external_libs/fmt' (https://github.com/fmtlib/fmt.git) registered for path 'external_libs/fmt'
Cloning into '/content/LightGBM/external_libs/compute'...
Cloning into '/content/LightGBM/external_libs/eigen'...
Cloning into '/content/LightGBM/external_libs/fast_double_parser'...
Cloning into '/content/LightGBM/external_libs/fmt'...
Submodule 'benchmark/dependencies/abseil-cpp' (https://github.com/abseil/abseil-cpp.git) registered for path 'external_libs/fast_double_parser/benchmarks/dependencies/abseil-cpp'
Submodule 'benchmark/dependencies/double-conversion' (https://github.com/google/doub

In [16]:
import lightgbm as lgb
print(lgb.__version__)

4.6.0


In [17]:
cat_feature_indexes = [X_train.columns.get_loc(col) for col in X_train.select_dtypes(include='category').columns]

In [18]:
cat_feature_indexes

[2, 4, 5]

In [19]:
lgb_clf = lgb.LGBMClassifier(
    max_depth=3,
    n_estimators=10,
    learning_rate=0.1,
    random_state=42,
)

lgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], categorical_feature=cat_feature_indexes)

train_proba_lgb = lgb_clf.predict_proba(X_train)[:, 1]
val_proba_lgb   = lgb_clf.predict_proba(X_val)[:, 1]

train_auroc_lgb = roc_auc_score(y_train, train_proba_lgb)
val_auroc_lgb   = roc_auc_score(y_val,   val_proba_lgb)

print(f"LightGBM — Train AUROC: {train_auroc_lgb:.4f}")
print(f"LightGBM — Val   AUROC: {val_auroc_lgb:.4f}")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 2442, number of negative: 9558
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002243 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1826
[LightGBM] [Info] Number of data points in the train set: 12000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203500 -> initscore=-1.364561
[LightGBM] [Info] Start training from score -1.364561
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

Модель не погана, але в порівнянні з XGBoostClassifier метрика показала менші показники. Можливо через слабкі параметри.

5. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `LightGBM` з лекції знайдіть оптимальні значення гіперпараметрів `LightGBM` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **10**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. І тут ми також ставимо loss - негативне значення AUROC, як і при пошуці гіперпараметрів для XGBoost. До речі, можна спробувати написати код так, аби в objective передавати лише модель і не писати схожий код двічі :)

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_lgb_clf` модель `LightGBM` з найкращими гіперпараметрами
    - навчіть модель `final_lgb_clf`
    - оцініть якість моделі `final_lgb_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (4) цього завдання?

In [20]:
def objective(params):
    clf = lgb.LGBMClassifier(
        n_estimators=int(params['n_estimators']),
        learning_rate=params['learning_rate'],
        max_depth=int(params['max_depth']),
        num_leaves=int(params['num_leaves']),
        min_child_weight=params['min_child_weight'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        min_split_gain=params['min_split_gain'],
        random_state=42,
        verbose=-1,
    )

    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=cat_feature_indexes,
    )

    pred = clf.predict_proba(X_val)[:, 1]
    rocauc = roc_auc_score(y_val, pred)
    return {'loss': -rocauc, 'status': STATUS_OK}

space = {
    'n_estimators':    hp.quniform('n_estimators', 50, 500, 25),
    'learning_rate':   hp.uniform('learning_rate', 0.01, 0.3),
    'max_depth':       hp.quniform('max_depth', 3, 15, 1),
    'num_leaves':      hp.quniform('num_leaves', 20, 150, 1),
    'min_child_weight':hp.quniform('min_child_weight', 1, 10, 1),
    'subsample':       hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree':hp.uniform('colsample_bytree', 0.5, 1.0),
    'reg_alpha':       hp.uniform('reg_alpha', 0, 1),
    'reg_lambda':      hp.uniform('reg_lambda', 0, 1),
    'min_split_gain':  hp.uniform('min_split_gain', 0, 0.1),
}

trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=10, trials=trials)

best['n_estimators']     = int(best['n_estimators'])
best['max_depth']        = int(best['max_depth'])
best['num_leaves']       = int(best['num_leaves'])
best['min_child_weight'] = int(best['min_child_weight'])

print("Найкращі гіперпараметри:", best)

final_lgb_clf = lgb.LGBMClassifier(
    n_estimators=best['n_estimators'],
    learning_rate=best['learning_rate'],
    max_depth=best['max_depth'],
    num_leaves=best['num_leaves'],
    min_child_weight=best['min_child_weight'],
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda'],
    min_split_gain=best['min_split_gain'],
    random_state=42,
    verbose=-1,
)

final_lgb_clf.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_feature_indexes,
)

train_proba_lgb = final_lgb_clf.predict_proba(X_train)[:, 1]  # ← predict_proba
val_proba_lgb   = final_lgb_clf.predict_proba(X_val)[:, 1]

final_train_auroc_lgb = roc_auc_score(y_train, train_proba_lgb)
final_val_auroc_lgb   = roc_auc_score(y_val,   val_proba_lgb)

print(f"Final LightGBM — Train AUROC: {final_train_auroc_lgb:.4f}")
print(f"Final LightGBM — Val   AUROC: {final_val_auroc_lgb:.4f}")
print(f"Порівняно з базовим LightGBM — Val AUROC: {final_val_auroc_lgb - val_auroc_lgb:.4f}")

100%|██████████| 10/10 [00:12<00:00,  1.23s/trial, best loss: -0.9354557925783662]
Найкращі гіперпараметри: {'colsample_bytree': np.float64(0.6166231851047222), 'learning_rate': np.float64(0.09330864496201378), 'max_depth': 3, 'min_child_weight': 9, 'min_split_gain': np.float64(0.07587869430817407), 'n_estimators': 225, 'num_leaves': 134, 'reg_alpha': np.float64(0.216938054538648), 'reg_lambda': np.float64(0.7852484164729767), 'subsample': np.float64(0.6541487906468145)}
Final LightGBM — Train AUROC: 0.9676
Final LightGBM — Val   AUROC: 0.9355
Порівняно з базовим LightGBM — Val AUROC: 0.0192


6. Оберіть модель з експериментів в цьому ДЗ і зробіть новий `submission` на Kaggle та додайте код для цього і скріншот скора на публічному лідерборді.
  
  **Напишіть коментар, чому ви обрали саме цю модель?**

  І я вас вітаю - це останнє завдання з цим набором даних 💪 На цьому етапі корисно проаналізувати, які моделі показали себе найкраще і подумати, чому.

In [22]:
results_df = pd.DataFrame(results).T.round(4)
results_df['Val-Train gap'] = (results_df['Val AUROC'] - results_df['Train AUROC']).round(4)
print(results_df.to_string())

                  Train AUROC  Val AUROC  Val-Train gap
XGBoost базовий        0.9615     0.9343        -0.0272
XGBoost final          0.9661     0.9342        -0.0319
LightGBM базовий       0.9181     0.9162        -0.0019
LightGBM final         0.9676     0.9355        -0.0321


На Kaggle, як я зрозуміла, я не приймаю участь, тому що з минулого потоку.

Обираю модель LightGBM. Після підбору гіперпараметрів показала кращі результати. Хоча і XGBoost базовий теж дав класні показники на самому початку.